# Notebook to preprocess IFRC reports

## Steps
1. Load reports text from JSON (check it is the correct version, eventually redo scraping ourself)
2. Filter out unnessecary reports
3. Clean text
4. Separate sentences and tokenize
5. Add hazard category for each report (use Laura's reclassifying)
6. Add division according to header


In [1]:
import pandas as pd
import json
from collections import Counter
from src.text_processing_functions import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import copy as cp
import spacy
import regex as re
from src.data import *
from src.hazard_def import hazard_subtype_kw_searc
#from spacy.language import Language  # For custom pipeline components
#from spacy_langdetect import LanguageDetector  # For language detection
import spacy_fastlang

In [2]:
file_path = DATA_IN_JSONS + '/filtered_report_types_nat_hazards_v3.json' #not sure if this is the correct file

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    all_ifrc_reports_info_unnested = json.load(json_file)

In [3]:
# Convert the JSON data into a Pandas DataFrame
data = pd.DataFrame(all_ifrc_reports_info_unnested)

## Filtering useless reports

In [4]:
## filter out useless reports
filtered_reports = [
    disaster_report for disaster_report in all_ifrc_reports_info_unnested
    if disaster_report['appealType'] in ['Operations Update', 'DREF Operation', 'DREF Operation Final Report', 'DREF Operation Update']
]

## Text preprocessing

In [5]:
#load libraries fo nlp
#not clear exactly which preprocessing steps must be undertaken
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')  # Download sentence tokenizer
nltk.download('stopwords') # Download stopwords

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:

#https://spacy.io/universe/project/spacy_fastlang
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("language_detector")

#or custom from https://medium.com/@j.boldsen.ryan/detecting-languages-with-spacy-and-spacy-langdetect-0b733a2a06d2
#from spacy.language import Language  # For custom pipeline components
#from spacy_langdetect import LanguageDetector  # For language detection
# Custom language detector factory function
#@Language.factory("language_detector")
#def create_language_detector(nlp, name):
#    return LanguageDetector() # Create the detector component

# Add language detector to the spaCy pipeline
#nlp.add_pipe("language_detector", last=True)

# Function to check if the text is in English
def detect_language(text):
    doc = nlp(text)
    return doc._.language  # Check if detected language is English

def rewrite_flood_affect(text):
    text, _ = re.subn(r'\b(ood.*)', "flood", text)
    text, _ = re.subn(r'\b(aect.*)', "affect", text)
    return text


In [ ]:
#clean text and tokenize into sentences
format_numbers = False
std_units = False
not_eng = []
for item in filtered_reports[:]:
    if 'text' in item:
        #identify language and get rid of reports not in english
        item['language'] = detect_language(item['text'])
        if item['language'] != 'en':
            filtered_reports.remove(item)
            not_eng.append(item)
            continue
        item['text_processed'] = clean_text(item['text'], format_numbers=format_numbers)
        item['text_processed'] = rewrite_flood_affect(item['text_processed'])
        if std_units:
            item['text_processed'] = standardize_units(item['text_processed'])
        item['sentences'] = sent_tokenize(item['text_processed'])

    else: # drop reports without text
        filtered_reports.remove(item)

In [8]:
import regex as re
#check for missing "ff" and "fl characters"
missing_chars = []
for report in filtered_reports[:]:
    if re.search(r'\b(ood.*)', report['text_processed']) or re.search(r'\b(aect.*)', report['text_processed']):
        missing_chars.append(report)

print(len(missing_chars))

0


In [9]:
from src.text_processing_functions import *
test = "\nThe dual crises of flooding and cholera outbreaks struck at a time when 24.8 million people in Sudan needed humanitarian assistance,\nover 10 million were internally displaced, and the country faced one of the worst food security crises in the world."
test = "24.8 million people affected in seven forks dam"
#test_df = data.where(data.appealCode.isin(['MDRSD034'])).dropna(how='all').iloc[0].text
clean_text(test, format_numbers=True)


'24800000.0 people affected in 7.0 forks dam'

In [10]:
# Add the ISO code to each dict in the list

import pycountry

for report in filtered_reports[:]:
    country_name = report.get("location")
    try:
        # Lookup the ISO code using pycountry
        country = pycountry.countries.get(name=country_name)
        if country:
            report["iso_code"] = country.alpha_3  # Adds the ISO 3166-1 Alpha-3 code
        else:
            report["iso_code"] = "Unknown"
    except KeyError:
        report["iso_code"] = "Unknown"


## Add natural hazard type and filter out other disasters

In [11]:
# Apply the function to the 'text_preprocessed' column of the DataFrame
for report in filtered_reports[:]:
    report['hazards_found_kw'] = check_hazard_type_keyword(report['text_processed'], hazard_subtype_kw_searc)

In [12]:
# filter out reports with no identified hazard with keyword searc but keep those where disasterType is related to a nat haz
filtered_reports_hazonly = cp.deepcopy(filtered_reports)
for report in filtered_reports_hazonly[:]:
    if (len(report['hazards_found_kw']) == 0):# and (report['disasterTypeReclassified'] not in disasterType_nathaz)):
        filtered_reports_hazonly.remove(report)

In [13]:
joined_df = pd.concat([pd.DataFrame(filtered_reports).groupby('disasterTypeReclassified').count()['reportName'], pd.DataFrame(filtered_reports_hazonly).groupby('disasterTypeReclassified').count()['reportName']],
                      axis=1, keys=["all", "hazonly"])

In [14]:
joined_df.sum()

all        1875
hazonly    1840
dtype: int64

In [15]:
# joined_df.plot(kind='bar', figsize=(10, 6))

## Select subsections containing natural hazard info

In [16]:
for report in filtered_reports_hazonly[:]:
    report['nathaz_text'] = select_hazard_description(report['sentences'])

In [17]:
#also for all haz
for report in filtered_reports[:]:
    report['nathaz_text'] = select_hazard_description(report['sentences'])

## Save data

In [18]:
fname_nathaz = 'nathaz_ifrc_reports_info_processed' #not sure if this is the correct file
if format_numbers:
    fname_nathaz = fname_nathaz + '_format_nb'
if std_units:
    fname_nathaz = fname_nathaz + '_std_units'
with open(DATA_IN_JSONS +fname_nathaz+'.json', 'w') as f:
    json.dump(filtered_reports_hazonly, f, indent=4)

In [19]:
fname_all = 'all_ifrc_reports_info_processed_extended' #not sure if this is the correct file
if format_numbers:
    fname_all = fname_all + '_format_nb'
if std_units:
    fname_all = fname_all + '_std_units'
with open(DATA_IN_JSONS +fname_all+".json", 'w') as f:
    json.dump(filtered_reports, f, indent=4)

In [20]:
fname_all

'all_ifrc_reports_info_processed_extended'